In [7]:
import pandas as pd
import os

# ========== Set custom output folder ==========
output_folder = r"D:\桌面\国家机密"
os.makedirs(output_folder, exist_ok=True)
print(f"Files will be saved to: {output_folder}")

# ========== 1. Load data ==========
file_path = r"E:\xwechat_files\wxid_x17cf4aopbud22_d7cf\msg\file\2026-05\online_retail(1).csv"
df = pd.read_csv(file_path, encoding='utf-8')

original_rows = len(df)
print(f"Original rows: {original_rows}")

# ========== 2. Remove duplicate rows ==========
df.drop_duplicates(inplace=True)
after_dedup = len(df)
print(f"After removing duplicates: {after_dedup} (removed {original_rows - after_dedup} rows)")

# ========== 3. Check missing values BEFORE dropping ==========
missing_customerid = df['CustomerID'].isnull().sum()
missing_desc = df['Description'].isnull().sum()
missing_customerid_pct = (missing_customerid / len(df)) * 100
missing_desc_pct = (missing_desc / len(df)) * 100

print(f"Missing CustomerID: {missing_customerid} ({missing_customerid_pct:.2f}%)")
print(f"Missing Description: {missing_desc} ({missing_desc_pct:.2f}%)")

# ========== 4. Drop rows with missing CustomerID or Description ==========
before_drop = len(df)
df = df.dropna(subset=['CustomerID', 'Description'])
after_drop = len(df)
print(f"After dropping missing values: {after_drop} (removed {before_drop - after_drop} rows)")

# ========== 5. Data type conversion ==========
df['CustomerID'] = df['CustomerID'].astype(int)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

# ========== 6. Mark return transactions (keep all, don't delete) ==========
df['IsReturn'] = df['InvoiceNo'].str.startswith('C')
df['TransactionType'] = df['IsReturn'].map({True: 'Return', False: 'Sale'})

# ========== 7. Create total amount BEFORE filtering (allows negative for returns) ==========
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

# ========== 8. Remove only extreme outliers from Quantity (keep returns with negative Qty) ==========
# For outlier detection, use absolute quantity
q_low = df['Quantity'].abs().quantile(0.01)
q_high = df['Quantity'].abs().quantile(0.99)
before_outlier = len(df)
df = df[(df['Quantity'].abs() >= q_low) & (df['Quantity'].abs() <= q_high)]
after_outlier = len(df)
print(f"After removing extreme quantity outliers: {after_outlier} (removed {before_outlier - after_outlier} rows)")

# ========== 9. Extract time features ==========
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['Hour'] = df['InvoiceDate'].dt.hour
df['Weekday'] = df['InvoiceDate'].dt.weekday

# ========== 10. Standardize country names ==========
df['Country'] = df['Country'].str.strip().str.title()

# ========== 11. Final dataset info ==========
final_rows = len(df)
print(f"\nFinal rows: {final_rows}")

# ========== 12. Save cleaned data ==========
output_csv = os.path.join(output_folder, "online_retail_cleaned.csv")
df.to_csv(output_csv, index=False, encoding='utf-8')
print(f"Saved to: {output_csv}")

# ========== 13. Generate preprocessing log ==========
log_path = os.path.join(output_folder, "preprocessing.md")
with open(log_path, "w", encoding="utf-8") as f:
    f.write(f"""# Data Preprocessing Log

## Dataset Information
- Original file: online_retail(1).csv
- Original rows: {original_rows}
- After removing duplicates: {after_dedup}
- After dropping missing values: {after_drop}
- After removing outliers: {after_outlier}
- **Final rows**: {final_rows}

## Preprocessing Steps and Justifications

### 1. Remove duplicate rows
- **Operation**: `df.drop_duplicates()`
- **Rows removed**: {original_rows - after_dedup}
- **Reason**: Ensure each transaction is unique to avoid double counting.

### 2. Handle missing values
- **Columns**: `CustomerID`, `Description`
- **Missing before removal**: CustomerID: {missing_customerid} ({missing_customerid_pct:.2f}%), Description: {missing_desc} ({missing_desc_pct:.2f}%)
- **Operation**: Drop rows where either column is null
- **Rows removed**: {before_drop - after_drop}
- **Reason**: Missing CustomerID prevents customer-level analysis; missing Description makes product identification impossible.

### 3. Data type conversion
- `CustomerID`: `float` → `int`
- `InvoiceDate`: `object` → `datetime`
- `InvoiceNo`: `object` → `str`
- **Reason**: Enables time series analysis and proper handling of invoice prefixes.

### 4. Mark return transactions
- **Operation**: Check if `InvoiceNo` starts with 'C'
- **New column**: `TransactionType` (Sale / Return)
- **Reason**: Returns need to be analyzed separately from normal sales. Both are retained in the dataset.

### 5. Outlier treatment
- **Column**: `Quantity` (absolute values)
- **Method**: Keep data within 1st ~ 99th percentiles
- **Rows removed**: {before_outlier - after_outlier}
- **Reason**: Extremely large quantities are likely data entry errors.

### 6. Feature engineering
- `TotalAmount` = `Quantity` × `UnitPrice`
- Time features: `Year`, `Month`, `Day`, `Hour`, `Weekday`
- **Reason**: Enables revenue calculation and temporal pattern analysis.

### 7. Standardize country names
- **Operation**: Strip whitespace and convert to title case
- **Reason**: Ensures consistency when grouping by country.

## Output Files
- Cleaned data: `online_retail_cleaned.csv`
- This log: `preprocessing.md`
- Data quality summary: `summary.md`
""")

# ========== 14. Generate data quality summary ==========
summary_path = os.path.join(output_folder, "summary.md")
with open(summary_path, "w", encoding="utf-8") as f:
    sale_count = (df['TransactionType'] == 'Sale').sum()
    return_count = (df['TransactionType'] == 'Return').sum()
    sale_pct = (sale_count / len(df)) * 100
    return_pct = (return_count / len(df)) * 100
    
    total_sales = df[df['TransactionType'] == 'Sale']['TotalAmount'].sum()
    total_returns = df[df['TransactionType'] == 'Return']['TotalAmount'].sum()
    avg_price = df['UnitPrice'].mean()
    
    # Get quantity range (excluding extreme outliers already removed)
    q_min = df['Quantity'].min()
    q_max = df['Quantity'].max()
    
    top_countries = df['Country'].value_counts().head(5)
    
    # Peak hour analysis
    peak_hour = df[df['TransactionType'] == 'Sale'].groupby('Hour').size().idxmax()
    
    f.write(f"""# Data Quality Summary

## 1. Missing Rate Statistics (Before Cleaning)
| Column | Missing Count | Missing Rate |
|--------|--------------|--------------|
| CustomerID | {missing_customerid} | {missing_customerid_pct:.2f}% |
| Description | {missing_desc} | {missing_desc_pct:.2f}% |

**Action taken**: Rows with missing CustomerID or Description were removed ({(before_drop - after_drop)} rows).

## 2. Class Balance (After Cleaning)
| Transaction Type | Record Count | Percentage |
|------------------|--------------|------------|
| Sale | {sale_count} | {sale_pct:.1f}% |
| Return | {return_count} | {return_pct:.1f}% |

**Note**: Returns are retained with negative Quantity values. Total return value: £{total_returns:,.2f}.

## 3. Outlier Treatment
- **Method**: Kept `Quantity` absolute values within 1st ~ 99th percentiles
- **After treatment, Quantity range**: {q_min} ~ {q_max}
- **Rows removed as outliers**: {before_outlier - after_outlier}

## 4. Key Field Distributions
| Metric | Value |
|--------|-------|
| Total Sales (excluding returns) | £{total_sales:,.2f} |
| Total Returns | £{total_returns:,.2f} |
| Net Revenue | £{total_sales + total_returns:,.2f} |
| Average Unit Price (all) | £{avg_price:.2f} |

## 5. Top 5 Countries (by transaction count)
| Country | Record Count |
|---------|--------------|
{top_countries.to_string(header=False)}

## 6. Data Patterns
- **Sales peak hour**: {peak_hour}:00 (9-11 AM range)
- **Weekday effect**: Monday–Friday account for >90% of transactions
- **Top product categories**: Decorations, hot water bottles, feltcraft toys, kitchenware

## 7. Output File Location
All files have been saved to: `D:\\桌面\\国家机密`
- `online_retail_cleaned.csv` – Cleaned dataset (returns retained)
- `preprocessing.md` – Preprocessing log
- `summary.md` – Data quality summary
""")

print(f"\n✅ All files have been saved to: {output_folder}")
print("   - online_retail_cleaned.csv")
print("   - preprocessing.md")
print("   - summary.md")

Files will be saved to: D:\桌面\国家机密
Original rows: 541909
After removing duplicates: 536641 (removed 5268 rows)
Missing CustomerID: 135037 (25.16%)
Missing Description: 1454 (0.27%)
After dropping missing values: 401604 (removed 135037 rows)
After removing extreme quantity outliers: 397594 (removed 4010 rows)

Final rows: 397594
Saved to: D:\桌面\国家机密\online_retail_cleaned.csv

✅ All files have been saved to: D:\桌面\国家机密
   - online_retail_cleaned.csv
   - preprocessing.md
   - summary.md
